# PM3 - Tratamento de Dados

Notebook reprodutível do Projeto Mensal 3 usando dados reais do SIM/DATASUS.


## Fonte dos dados
- Sistema de Informação sobre Mortalidade (SIM): https://dadosabertos.saude.gov.br/dataset/sim
- Data de acesso: 16/05/2026


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

cwd = Path.cwd().resolve()
if (cwd / 'dados_brutos').exists():
    base_dir = cwd
elif cwd.name == 'notebooks' and (cwd.parent / 'dados_brutos').exists():
    base_dir = cwd.parent
else:
    base_dir = Path('..').resolve()
raw_path = base_dir / 'dados_brutos' / 'dadosSIM2005-.csv'
df = pd.read_csv(raw_path)
df.shape, df.head()


In [ ]:
# Diagnóstico inicial de qualidade
diagnostico = pd.DataFrame({
    'tipo': ['linhas', 'colunas', 'duplicatas', 'faltantes'],
    'valor': [len(df), df.shape[1], df.duplicated().sum(), df.isna().sum().sum()]
})
diagnostico


In [ ]:
# Execução do pipeline completo
import sys
sys.path.append(str(base_dir))
import projMens3
projMens3.main()


In [ ]:
final = pd.read_csv(base_dir / 'dados_tratados' / 'dataset_final_tratado.csv', sep=';')
final.head(), final.shape


In [ ]:
# Estatísticas descritivas e frequências categóricas
display(final[['idade', 'idade_meses', 'idade_normalizada_minmax', 'idade_padronizada_zscore']].describe())
display(final['grupo_causa'].value_counts().head(10))
display(final.groupby('ano').size().rename('total_obitos').reset_index())


## Cruzamentos para conclusão analítica

Aqui a análise deixa de olhar apenas frequências isoladas e cruza causa, faixa etária e local de ocorrência.


In [ ]:
cruzamento_causa_faixa = pd.crosstab(final['grupo_causa'], final['faixa_etaria_bi'])
cruzamento_causa_faixa['total_obitos'] = cruzamento_causa_faixa.sum(axis=1)
display(cruzamento_causa_faixa.sort_values('total_obitos', ascending=False).head(10))

causa_externa_por_faixa = final.groupby('faixa_etaria_bi')['causa_externa'].agg(['count', 'sum'])
causa_externa_por_faixa['percentual_causas_externas'] = (causa_externa_por_faixa['sum'] / causa_externa_por_faixa['count'] * 100).round(1)
display(causa_externa_por_faixa)


Conclusão: os menores de 1 ano concentram causas perinatais e malformações congênitas. Já entre 1 e 4 anos, o peso proporcional das causas externas aumenta bastante, o que muda a interpretação para BI e políticas de prevenção.


## Onde os gráficos são gerados

Ao executar `projMens3.main()`, os gráficos são salvos como imagens PNG em `documentacao/graficos/` e também entram no `documentacao/relatorio_final.pdf`.


In [ ]:
from IPython.display import Image, display

graph_dir = base_dir / 'documentacao' / 'graficos'
graficos = sorted(graph_dir.glob('*.png'))
print(f'{len(graficos)} gráficos gerados em: {graph_dir}')
for grafico in graficos:
    print(grafico.name)
    display(Image(filename=str(grafico)))
